# A County-Level Suitability Analysis for New Data Center Sites in the United States

In [15]:
import arcpy
from arcpy.sa import *
import os

data_dir = r".\data"
arcpy.env.workspace = data_dir

arcpy.env.overwriteOutput = True

arcpy.CheckOutExtension("Spatial")

'CheckedOut'

## 1. Dissolve state features into single contiguous feature

In [12]:
input_states = os.path.join(data_dir, "US_Contig_State_Boundaries", "US_Contig_State_Boundaries.shp")
us_contig_boundary_dir = os.path.join(data_dir, "US_Contiguous_Boundary")
contiguous_us = os.path.join(us_contig_boundary_dir, "US_Contiguous_Boundary.shp")
os.makedirs(us_contig_boundary_dir, exist_ok=True)

print("Dissolving 48 state boundaries into a single contiguous feature...")

arcpy.management.Dissolve(
    in_features=input_states,
    out_feature_class=contiguous_us
)

print(f"Success! Master boundary saved to: {contiguous_us}")

Dissolving 48 state boundaries into a single contiguous feature...
Success! Master boundary saved to: .\data\US_Contiguous_Boundary\US_Contiguous_Boundary.shp


## 2. Reproject Boundary to EPSG:5070 (NAD83 / Conus Albers)

In [13]:
conus_albers_sr = arcpy.SpatialReference(5070)

projected_contiguous_us = os.path.join(us_contig_boundary_dir, "US_Contiguous_Boundary_5070.shp")

print("Projecting boundary to EPSG:5070...")

arcpy.management.Project(
    in_dataset=contiguous_us,
    out_dataset=projected_contiguous_us,
    out_coor_system=conus_albers_sr
)

print(f"Success! Projected boundary saved to: {projected_contiguous_us}")

Projecting boundary to EPSG:5070...
Success! Projected boundary saved to: .\data\US_Contiguous_Boundary\US_Contiguous_Boundary_5070.shp


## 3. Clip and Standardize Slope Raster

In [ ]:
raw_slope_tif = os.path.join(data_dir, "slope_1KMmd_GMTEDmd.tif")
slope_processing_dir = os.path.join(data_dir, "Slope_Processing")
os.makedirs(slope_processing_dir, exist_ok=True)

clipped_slope_tif = os.path.join(slope_processing_dir, "Slope_Clipped_5070.tif")
standardized_slope_tif = os.path.join(slope_processing_dir, "Slope_Standardized.tif")

print("Extracting slope raster using the contiguous US boundary mask...")

with arcpy.EnvManager(outputCoordinateSystem=conus_albers_sr, snapRaster=raw_slope_tif):
    clipped_slope = ExtractByMask(
        in_raster=raw_slope_tif,
        in_mask_data=projected_contiguous_us
    )
    clipped_slope.save(clipped_slope_tif)

print("Excluding slopes > 10 degrees and scaling remaining areas from 0.0 to 1.0...")

# Map Algebra logic:
# 1. 'clipped_slope <= 10' acts as a conditional filter. Anything > 10 becomes NoData.
# 2. '1.0 - (clipped_slope / 10.0)' inverts and normalizes the scale.
#    A slope of 0 degrees becomes 1.0 (highly suitable).
#    A slope of 10 degrees becomes 0.0 (least suitable).

standardized_slope = Con(clipped_slope <= 10, 1.0 - (clipped_slope / 10.0))
standardized_slope.save(standardized_slope_tif)

print(f"Success! Standardized slope raster saved to: {standardized_slope_tif}")